# Настройка DuckLake

In [1]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [2]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml or /Users/i.korsakov/.jupysql/config.

Did not find user configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml.

In [3]:
%sql duckdb:///:memory:

Connecting and switching to connection 'duckdb:///:memory:'

# Создание подключения к DuckLake

In [4]:
%%sql
INSTALL ducklake;

,Success


In [5]:
%%sql
ATTACH 'ducklake:my_ducklake.ducklake' AS my_ducklake;
USE my_ducklake;

,Success


# Создание таблицы в DuckLake

In [6]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

,Success


In [7]:
%%sql
FROM fake_data

,id,name,email,city,country
0,1,Jayda Lesch,wilberterdman@jones.net,Gusikowskistad,Chile
1,2,Mina Dicki,sadieritchie@beahan.io,Marcellefort,Myanmar
2,3,Cleveland Corwin,ellsworthhagenes@haley.name,Hellerland,Bouvet Island
3,4,Nicole Feeney,sinceremayer@kub.com,Ronnyberg,Guam
4,5,Isac McDermott,fernandojohnston@bogan.biz,Swaniawskiville,Sao Tome and Principe
...,...,...,...,...,...
95,96,Kaitlin Gibson,katheryndeckow@batz.com,Roderickland,Faroe Islands
96,97,Noble Thompson,meganefadel@hessel.org,Cruickshankshire,Saint Helena
97,98,Naomi Kuhic,vincentziemann@langworth.io,Juniorshire,Israel
98,99,Dashawn Hermiston,tobygutmann@koelpin.net,McCulloughtown,Sri Lanka


# Спецификация DuckLake

[Specification/Tables](https://ducklake.select/docs/stable/specification/tables/overview)

![](https://ducklake.select/images/schema/ducklake-schema-v1.0-light.svg)

In [8]:
%%sql
SELECT
    table_catalog,
    table_name
FROM information_schema.tables

,table_catalog,table_name
0,__ducklake_metadata_my_ducklake,ducklake_column
1,__ducklake_metadata_my_ducklake,ducklake_column_mapping
2,__ducklake_metadata_my_ducklake,ducklake_column_tag
3,__ducklake_metadata_my_ducklake,ducklake_data_file
4,__ducklake_metadata_my_ducklake,ducklake_delete_file
5,__ducklake_metadata_my_ducklake,ducklake_files_scheduled_for_deletion
6,__ducklake_metadata_my_ducklake,ducklake_file_column_stats
7,__ducklake_metadata_my_ducklake,ducklake_file_partition_value
8,__ducklake_metadata_my_ducklake,ducklake_file_variant_stats
9,__ducklake_metadata_my_ducklake,ducklake_inlined_data_1_1


In [9]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [10]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [11]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,0,"created_schema:""main""",None,None,None
1,1,"created_table:""main"".""fake_data"",inserted_into...",None,None,None


In [12]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-05-28 11:41:17.231188+03:00,0,1,0
1,1,2026-05-28 11:41:17.253975+03:00,1,2,1


In [13]:
%%sql
FROM ducklake_data_file

,data_file_id,table_id,begin_snapshot,end_snapshot,file_order,path,path_is_relative,file_format,record_count,file_size_bytes,footer_size,row_id_start,partition_id,encryption_key,mapping_id,partial_max
0,0,1,1,<NA>,<NA>,ducklake-019e6dbe-bcc7-764d-a258-97e5e7ba109a....,True,parquet,100,7169,661,0,<NA>,None,<NA>,<NA>


In [14]:
df = %sql FROM ducklake_data_file WHERE end_snapshot IS NULL

In [15]:
pd.read_parquet(f'my_ducklake.ducklake.files/main/fake_data/{df.path[0]}')

,id,name,email,city,country
0,1,Jayda Lesch,wilberterdman@jones.net,Gusikowskistad,Chile
1,2,Mina Dicki,sadieritchie@beahan.io,Marcellefort,Myanmar
2,3,Cleveland Corwin,ellsworthhagenes@haley.name,Hellerland,Bouvet Island
3,4,Nicole Feeney,sinceremayer@kub.com,Ronnyberg,Guam
4,5,Isac McDermott,fernandojohnston@bogan.biz,Swaniawskiville,Sao Tome and Principe
...,...,...,...,...,...
95,96,Kaitlin Gibson,katheryndeckow@batz.com,Roderickland,Faroe Islands
96,97,Noble Thompson,meganefadel@hessel.org,Cruickshankshire,Saint Helena
97,98,Naomi Kuhic,vincentziemann@langworth.io,Juniorshire,Israel
98,99,Dashawn Hermiston,tobygutmann@koelpin.net,McCulloughtown,Sri Lanka


# Уборка в DuckLake
- [Expire Snapshots](https://ducklake.select/docs/stable/duckdb/maintenance/expire_snapshots)
- [Cleanup of Files](https://ducklake.select/docs/stable/duckdb/maintenance/cleanup_of_files)

In [16]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-05-28 11:41:17.231188+03:00,0,1,0
1,1,2026-05-28 11:41:17.253975+03:00,1,2,1


In [17]:
%%sql
CALL ducklake_expire_snapshots('my_ducklake', older_than => now() - INTERVAL '10 minute');

,Success


In [18]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-05-28 11:41:17.231188+03:00,0,1,0
1,1,2026-05-28 11:41:17.253975+03:00,1,2,1


In [19]:
%%sql
CALL ducklake_cleanup_old_files(
    'my_ducklake',
    cleanup_all => true
);

,Success


In [20]:
%%sql
SELECT current_catalog()

,current_catalog()
0,__ducklake_metadata_my_ducklake


# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [21]:
%%sql
USE 'my_ducklake';

,Success


In [22]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

,Success


In [23]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Jayda Lesch,wilberterdman@jones.net,Gusikowskistad,Chile,None
1,2,Mina Dicki,sadieritchie@beahan.io,Marcellefort,Myanmar,None
2,3,Cleveland Corwin,ellsworthhagenes@haley.name,Hellerland,Bouvet Island,None
3,4,Nicole Feeney,sinceremayer@kub.com,Ronnyberg,Guam,None
4,5,Isac McDermott,fernandojohnston@bogan.biz,Swaniawskiville,Sao Tome and Principe,None
...,...,...,...,...,...,...
95,96,Kaitlin Gibson,katheryndeckow@batz.com,Roderickland,Faroe Islands,None
96,97,Noble Thompson,meganefadel@hessel.org,Cruickshankshire,Saint Helena,None
97,98,Naomi Kuhic,vincentziemann@langworth.io,Juniorshire,Israel,None
98,99,Dashawn Hermiston,tobygutmann@koelpin.net,McCulloughtown,Sri Lanka,None


In [24]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

,Success


In [25]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Jayda Lesch,wilberterdman@jones.net,Gusikowskistad,Chile,Mr.
1,2,Mina Dicki,sadieritchie@beahan.io,Marcellefort,Myanmar,Ms.
2,3,Cleveland Corwin,ellsworthhagenes@haley.name,Hellerland,Bouvet Island,Ms.
3,4,Nicole Feeney,sinceremayer@kub.com,Ronnyberg,Guam,Ms.
4,5,Isac McDermott,fernandojohnston@bogan.biz,Swaniawskiville,Sao Tome and Principe,Dr.
...,...,...,...,...,...,...
95,96,Kaitlin Gibson,katheryndeckow@batz.com,Roderickland,Faroe Islands,Miss
96,97,Noble Thompson,meganefadel@hessel.org,Cruickshankshire,Saint Helena,Miss
97,98,Naomi Kuhic,vincentziemann@langworth.io,Juniorshire,Israel,Ms.
98,99,Dashawn Hermiston,tobygutmann@koelpin.net,McCulloughtown,Sri Lanka,Ms.


# Time travel

In [26]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [27]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [28]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,0,"created_schema:""main""",None,None,None
1,1,"created_table:""main"".""fake_data"",inserted_into...",None,None,None
2,2,altered_table:1,None,None,None
3,3,"inserted_into_table:1,deleted_from_table:1",None,None,None


In [29]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-05-28 11:41:17.231188+03:00,0,1,0
1,1,2026-05-28 11:41:17.253975+03:00,1,2,1
2,2,2026-05-28 11:41:20.580879+03:00,2,2,1
3,3,2026-05-28 11:41:20.744794+03:00,2,2,2


In [30]:
%%sql
USE 'my_ducklake';

,Success


In [33]:
%%sql
SELECT * FROM fake_data AT (VERSION => 3);

,id,name,email,city,country,name_prefix
0,1,Jayda Lesch,wilberterdman@jones.net,Gusikowskistad,Chile,Mr.
1,2,Mina Dicki,sadieritchie@beahan.io,Marcellefort,Myanmar,Ms.
2,3,Cleveland Corwin,ellsworthhagenes@haley.name,Hellerland,Bouvet Island,Ms.
3,4,Nicole Feeney,sinceremayer@kub.com,Ronnyberg,Guam,Ms.
4,5,Isac McDermott,fernandojohnston@bogan.biz,Swaniawskiville,Sao Tome and Principe,Dr.
...,...,...,...,...,...,...
95,96,Kaitlin Gibson,katheryndeckow@batz.com,Roderickland,Faroe Islands,Miss
96,97,Noble Thompson,meganefadel@hessel.org,Cruickshankshire,Saint Helena,Miss
97,98,Naomi Kuhic,vincentziemann@langworth.io,Juniorshire,Israel,Ms.
98,99,Dashawn Hermiston,tobygutmann@koelpin.net,McCulloughtown,Sri Lanka,Ms.


In [32]:
%sql rollback

,Success


## 